# GCI v18 — Alternative Direction Rank-Blend Probe

**Purpose:** create a cautious candidate that blends the existing v15 champion with a new two-model LightGBM rank blend. This is a *research probe*, not a validated leaderboard improvement.

## What was tested

On an independent 3-fold stratified validation split (seed `20261020`), the v18 simple-model blend gave pooled AUC **0.756907**, versus **0.756371** for the better single variant (`no_interactions`) and **0.756086** for `no_frequency`. Fold gains versus `no_interactions` were approximately `+0.000993`, `-0.000026`, and `+0.000600` (2/3 positive).

These validation results are from a lightweight standalone feature pipeline. They are **not directly comparable** to the v15 ensemble OOF score and do not prove that v18 improves the public leaderboard. The candidate below therefore uses only **5% v18 / 95% v15** as a conservative probe. Keep v15 as the fallback unless the platform score confirms an improvement.

## Inputs

Place `train.csv`, `test.csv`, `sample_submission.csv`, and `GCI_submission_v15_rankspace_consistent_view.csv` in the same data directory. The code uses `/content` in Colab when `train.csv` is there; otherwise it uses `/mnt/data`.


## Build two complementary LightGBM models and write the candidate

The full-data models use the same engineered feature definitions and fixed hyperparameters as the v18 experiment. The candidate file is named `GCI_submission_v18_alt_direction_probe.csv`.

In [ ]:
import os, warnings, time
import numpy as np, pandas as pd
from scipy.stats import rankdata
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
ROOT='/content' if os.path.exists('/content/train.csv') else '/mnt/data'
OUT=os.path.join(ROOT,'GCI_v18_alt_direction')
os.makedirs(OUT,exist_ok=True)
start=time.time()
print('Loading supplied data...',flush=True)
tr=pd.read_csv(ROOT+'/train.csv'); te=pd.read_csv(ROOT+'/test.csv')
y=tr.TARGET.to_numpy()

def build(df):
    X=df.drop(columns=['TARGET','SK_ID_CURR'],errors='ignore').copy()
    X['EMP_SENTINEL']=(X.DAYS_EMPLOYED==365243).astype('int8')
    X['DAYS_EMPLOYED_CLEAN']=X.DAYS_EMPLOYED.replace(365243,np.nan)
    X['AGE_YEARS']=-X.DAYS_BIRTH/365.25
    X['EMPLOY_YEARS']=-X.DAYS_EMPLOYED_CLEAN/365.25
    X['REG_YEARS']=-X.DAYS_REGISTRATION/365.25
    X['ID_YEARS']=-X.DAYS_ID_PUBLISH/365.25
    X['PHONE_YEARS']=-X.DAYS_LAST_PHONE_CHANGE/365.25
    eps=1e-3
    pairs=[('AMT_CREDIT','AMT_INCOME_TOTAL'),('AMT_ANNUITY','AMT_INCOME_TOTAL'),('AMT_ANNUITY','AMT_CREDIT'),('AMT_GOODS_PRICE','AMT_CREDIT'),('AMT_GOODS_PRICE','AMT_INCOME_TOTAL'),('AMT_CREDIT','AMT_ANNUITY')]
    for a,b in pairs: X[f'{a}_DIV_{b}']=X[a]/(X[b].abs()+eps)
    X['CREDIT_MINUS_GOODS']=X.AMT_CREDIT-X.AMT_GOODS_PRICE
    X['INCOME_PER_FAMILY']=X.AMT_INCOME_TOTAL/(X.CNT_FAM_MEMBERS.fillna(1)+eps)
    X['CHILDREN_PER_FAMILY']=X.CNT_CHILDREN/(X.CNT_FAM_MEMBERS.fillna(1)+eps)
    ext=['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']
    X['EXT_MEAN']=X[ext].mean(axis=1); X['EXT_MIN']=X[ext].min(axis=1); X['EXT_MAX']=X[ext].max(axis=1); X['EXT_STD']=X[ext].std(axis=1); X['EXT_PRODUCT']=X[ext].prod(axis=1,min_count=1)
    for i,a in enumerate(ext):
        for b in ext[i+1:]: X[f'{a}_MINUS_{b}']=X[a]-X[b]; X[f'{a}_MULT_{b}']=X[a]*X[b]
    bureau=['AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR']
    X['BUREAU_MISS_N']=X[bureau].isna().sum(axis=1); X['BUREAU_ZERO_N']=(X[bureau].fillna(0)==0).sum(axis=1); X['BUREAU_SUM']=X[bureau].fillna(0).sum(axis=1); X['BUREAU_LOGSUM']=np.log1p(X['BUREAU_SUM'].clip(lower=0))
    X['BUREAU_HOUR_PER_MON']=X[bureau[0]]/(X[bureau[1]]+1); X['BUREAU_MON_PER_YEAR']=X[bureau[1]]/(X[bureau[3]]+1); X['BUREAU_QRT_PER_YEAR']=X[bureau[2]]/(X[bureau[3]]+1)
    cr=X['AMT_CREDIT'].replace(0,np.nan); ann=X['AMT_ANNUITY']; goods=X['AMT_GOODS_PRICE']; ratio=ann/cr
    X['Q_RAW']=ratio; X['Q_ROUND_3']=ratio.round(3); X['Q_ROUND_4']=ratio.round(4)
    X['Q_FEE129']=((ann-129.5)/cr).round(4); X['FEE_RESIDUAL']=ann-(ratio.round(4)*cr)
    X['CREDIT_GOODS_RATIO']=cr/goods.replace(0,np.nan); X['K_ROUND_3']=X['CREDIT_GOODS_RATIO'].round(3); X['K_ROUND_4']=X['CREDIT_GOODS_RATIO'].round(4)
    X['ANNUITY_MINUS_FEE']=ann-129.5; X['PAYMENT_TERM']=cr/(ann+eps); X['FEE_SHARE']=129.5/(ann.abs()+eps)
    X['AGE_X_EXTMEAN']=X['AGE_YEARS']*X['EXT_MEAN']; X['EMP_X_EXTMEAN']=X['EMPLOY_YEARS']*X['EXT_MEAN']; X['CREDIT_INCOME_X_EXT']=X['AMT_CREDIT_DIV_AMT_INCOME_TOTAL']*X['EXT_MEAN']
    for c in X.columns.copy():
        if X[c].isna().any(): X[c+'_MISS']=X[c].isna().astype('int8')
    for c in X.select_dtypes(include='object').columns: X[c]=X[c].fillna('__MISSING__').astype('category')
    return X

print('Engineering features...',flush=True)
Xtr=build(tr); Xte=build(te)
# Ensure train/test share missingness indicators even when missingness occurs in only one split.
for c in sorted(set(Xtr.columns).intersection(Xte.columns)):
    if not c.endswith('_MISS') and (Xtr[c].isna().any() or Xte[c].isna().any()):
        miss=c+'_MISS'
        if miss not in Xtr.columns: Xtr[miss]=Xtr[c].isna().astype('int8')
        if miss not in Xte.columns: Xte[miss]=Xte[c].isna().astype('int8')
for c in Xtr.select_dtypes(include='category').columns:
    cats=pd.Index(pd.concat([Xtr[c].astype(str),Xte[c].astype(str)],ignore_index=True).unique())
    Xtr[c]=pd.Categorical(Xtr[c].astype(str),categories=cats)
    Xte[c]=pd.Categorical(Xte[c].astype(str),categories=cats)
cat_cols=['ORGANIZATION_TYPE','OCCUPATION_TYPE','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE','NAME_FAMILY_STATUS','NAME_HOUSING_TYPE']
for c in cat_cols:
    vals=pd.concat([tr[c].fillna('__MISSING__').astype(str),te[c].fillna('__MISSING__').astype(str)],ignore_index=True)
    counts=vals.value_counts()
    Xtr[c+'_FREQ']=tr[c].fillna('__MISSING__').astype(str).map(counts).to_numpy()
    Xte[c+'_FREQ']=te[c].fillna('__MISSING__').astype(str).map(counts).to_numpy()
base_cols=list(Xtr.columns)
inter=['AGE_X_EXTMEAN','EMP_X_EXTMEAN','CREDIT_INCOME_X_EXT']
freq=[c for c in base_cols if c.endswith('_FREQ')]
variants={'no_interactions':[c for c in base_cols if c not in inter], 'no_frequency':[c for c in base_cols if c not in freq]}
params=dict(n_estimators=900,learning_rate=0.035,num_leaves=31,max_depth=-1,min_child_samples=120,max_bin=127,subsample=0.85,subsample_freq=1,colsample_bytree=0.85,reg_lambda=3.0,reg_alpha=0.05,objective='binary',verbosity=-1,n_jobs=4)
preds={}
for i,(name,cols) in enumerate(variants.items(),1):
    print(f'Training full model {i}/2: {name}, {len(cols)} features',flush=True)
    model=LGBMClassifier(**params,random_state=8100+i)
    model.fit(Xtr[cols],y,categorical_feature='auto')
    p=model.predict_proba(Xte[cols])[:,1]
    preds[name]=p.astype('float64')
    np.save(os.path.join(OUT,f'{name}_test_prediction.npy'),preds[name])
    print(' done; prediction range',float(p.min()),float(p.max()),'elapsed',round(time.time()-start,1),'s',flush=True)

v18_rank=(rankdata(preds['no_interactions'])+rankdata(preds['no_frequency']))/2
v18_rank=(v18_rank-v18_rank.min())/(v18_rank.max()-v18_rank.min())
v15=pd.read_csv(ROOT+'/GCI_submission_v15_rankspace_consistent_view.csv')
sample=pd.read_csv(ROOT+'/sample_submission.csv')
assert v15.SK_ID_CURR.equals(sample.SK_ID_CURR), 'v15 IDs do not match sample order'
assert te.SK_ID_CURR.equals(sample.SK_ID_CURR), 'test IDs do not match sample order'
v15_rank=rankdata(v15.TARGET.to_numpy())
v15_rank=(v15_rank-v15_rank.min())/(v15_rank.max()-v15_rank.min())
# Conservative experimental probe only: 5% V18 rank, 95% champion rank.
candidate=0.95*v15_rank+0.05*v18_rank
out=pd.DataFrame({'SK_ID_CURR':sample.SK_ID_CURR,'TARGET':candidate})
path=ROOT+'/GCI_submission_v18_alt_direction_probe.csv'
out.to_csv(path,index=False)
print('Wrote',path,'rows',len(out),'elapsed',round(time.time()-start,1),'s',flush=True)
print('Important: 5% blend is a probe, not a validated champion improvement.',flush=True)


## Acceptance rule

- Do **not** call this a confirmed improvement from the local validation alone.
- Validate the output's row count, IDs, columns, and finite predictions before submission.
- Compare the public leaderboard result with the current v15 score of **0.77891**.
- Promote v18 only if the actual leaderboard result is higher; otherwise retain v15.
